In [1]:
import pandas as pd


df = pd.read_parquet('../Imdb_Movie_Dataset.parquet')
df_aux = df.copy()

In [2]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
import numpy as np
import pandas as pd
import pickle
import gzip
import gc

df_aux['runtime'] = pd.to_numeric(df_aux['runtime'], errors='coerce')
df_aux['budget'] = pd.to_numeric(df_aux['budget'], errors='coerce')
df_aux['vote_average'] = pd.to_numeric(df_aux['vote_average'], errors='coerce')
df_aux['vote_count'] = pd.to_numeric(df_aux['vote_count'], errors='coerce')

df_clean = df_aux[
    df_aux['vote_average'].notna() & 
    (df_aux['vote_average'] > 0) & 
    df_aux['vote_count'].notna() &
    (df_aux['vote_count'] >= 5)
].copy()

print(f"Filmes com relevância estatística para treinamento (>= 5 votos): {len(df_clean)}")

cpi_history = {
    1913: 9.9,   1914: 10.0,  1915: 10.1,  1916: 10.9,  1917: 12.8,  1918: 15.1,  1919: 17.3,
    1920: 20.0,  1921: 17.9,  1922: 16.8,  1923: 17.1,  1924: 17.1,  1925: 17.5,  1926: 17.7,
    1927: 17.4,  1928: 17.1,  1929: 17.1,  1930: 16.7,  1931: 15.2,  1932: 13.7,  1933: 13.0,
    1934: 13.4,  1935: 13.7,  1936: 13.9,  1937: 14.4,  1938: 14.1,  1939: 13.9,  1940: 14.0,
    1941: 14.7,  1942: 16.3,  1943: 17.3,  1944: 17.6,  1945: 18.0,  1946: 19.5,  1947: 22.3,
    1948: 24.1,  1949: 23.8,  1950: 24.1,  1951: 26.0,  1952: 26.5,  1953: 26.7,  1954: 26.9,
    1955: 26.8,  1956: 27.2,  1957: 28.1,  1958: 28.9,  1959: 29.1,  1960: 29.6,  1961: 29.9,
    1962: 30.2,  1963: 30.6,  1964: 31.0,  1965: 31.5,  1966: 32.4,  1967: 33.4,  1968: 34.8,
    1969: 36.7,  1970: 38.8,  1971: 40.5,  1972: 41.8,  1973: 44.4,  1974: 49.3,  1975: 53.8,
    1976: 56.9,  1977: 60.6,  1978: 65.2,  1979: 72.6,  1980: 82.4,  1981: 90.9,  1982: 96.5,
    1983: 99.6,  1984: 103.9, 1985: 107.6, 1986: 109.6, 1987: 113.6, 1988: 118.3, 1989: 124.0,
    1990: 130.7, 1991: 136.2, 1992: 140.3, 1993: 144.5, 1994: 148.2, 1995: 152.4, 1996: 156.9,
    1997: 160.5, 1998: 163.0, 1999: 166.6, 2000: 172.2, 2001: 177.1, 2002: 179.9, 2003: 184.0,
    2004: 188.9, 2005: 195.3, 2006: 201.6, 2007: 207.34, 2008: 215.30, 2009: 214.54, 2010: 218.06,
    2011: 224.94, 2012: 229.59, 2013: 232.96, 2014: 236.74, 2015: 237.02, 2016: 240.01, 2017: 245.12,
    2018: 251.11, 2019: 255.66, 2020: 258.81, 2021: 270.97, 2022: 292.66, 2023: 304.70, 2024: 313.20,
    2025: 320.10, 2026: 326.50
}

cpi_2026 = cpi_history[2026]

def ajustar_orcamento(row):
    ano = row['release_year']
    orcamento = row['budget']
    if orcamento <= 0:
        return 0
    cpi_ano = cpi_history.get(ano)
    if not cpi_ano:
        ano_proximo = min(cpi_history.keys(), key=lambda x: abs(x - ano))
        cpi_ano = cpi_history[ano_proximo]
    multiplicador = cpi_2026 / cpi_ano
    return orcamento * multiplicador

df_clean['release_year'] = pd.to_datetime(df_clean['release_date'], errors='coerce').dt.year
median_year = df_clean['release_year'].median()
df_clean['release_year'] = df_clean['release_year'].fillna(median_year).astype(int)

df_clean['release_5_years'] = (df_clean['release_year'] // 5) * 5

df_clean['movie_age'] = 2027 - df_clean['release_year']

df_clean['has_budget'] = (df_clean['budget'] > 0).astype('int8')
df_clean['budget'] = df_clean['budget'].fillna(0)
df_clean['budget'] = df_clean.apply(ajustar_orcamento, axis=1)

df_clean['overview_len'] = df_clean['overview'].astype(str).fillna('').str.len()
df_clean['tagline_len'] = df_clean['tagline'].astype(str).fillna('').str.len()

df_clean['keywords'] = df_clean['keywords'].astype(str).fillna('')
df_clean['is_short_keyword'] = df_clean['keywords'].str.contains('short', case=False, regex=False).astype('int8')

df_clean['runtime'] = df_clean['runtime'].fillna(df_clean['runtime'].median())

df_clean['genres'] = df_clean['genres'].astype(str).fillna('')
df_clean['main_genre'] = df_clean['genres'].str.split(', ').str[0]

global_vote_mean = df_clean['vote_average'].mean()
genre_stats = df_clean.groupby('main_genre')['vote_average'].agg(['mean', 'count'])

smoothing_g = 15
df_clean['genre_vote_mean'] = df_clean['main_genre'].map(
    lambda x: (genre_stats.loc[x, 'mean'] * genre_stats.loc[x, 'count'] + global_vote_mean * smoothing_g) / (genre_stats.loc[x, 'count'] + smoothing_g) if x in genre_stats.index else global_vote_mean
)

genre_decade_stats = df_clean.groupby(['main_genre', 'release_5_years'])['vote_average'].agg(['mean', 'count'])
smoothing_gd = 20
def get_smooth_genre_decade_vote(row):
    key = (row['main_genre'], row['release_5_years'])
    if key in genre_decade_stats.index:
        stat = genre_decade_stats.loc[key]
        return (stat['mean'] * stat['count'] + row['genre_vote_mean'] * smoothing_gd) / (stat['count'] + smoothing_gd)
    return row['genre_vote_mean']

df_clean['genre_decade_vote_mean'] = df_clean.apply(get_smooth_genre_decade_vote, axis=1)

df_clean['log_vote_count'] = np.log1p(df_clean['vote_count'])
df_clean['votes_per_year'] = df_clean['vote_count'] / (df_clean['movie_age'] + 1)
df_clean['is_en'] = (df_clean['original_language'] == 'en').astype('int8')

numerical_features = [
    'runtime', 'vote_count', 'log_vote_count', 'release_year', 'release_5_years', 'movie_age',
    'budget', 'has_budget', 'is_short_keyword', 'overview_len', 'tagline_len', 
    'genre_vote_mean', 'genre_decade_vote_mean', 'has_budget', 'is_en', 'votes_per_year'
]
multi_value_categorical_features = ['genres', 'production_companies', 'production_countries']

all_base_features = numerical_features + multi_value_categorical_features
df_train_local = df_clean[all_base_features + ['vote_average']].copy()

top_n_categories = 50
cols_to_drop = ['main_genre'] if 'main_genre' in df_train_local.columns else []
new_features_dict = {}

for col in multi_value_categorical_features:
    df_train_local[col] = df_train_local[col].astype(str).fillna('')
    item_counts = df_train_local[col].str.split(', ').explode().str.strip().value_counts()
    top_items = item_counts[item_counts.index != ''].head(top_n_categories).index.tolist()
    
    for item_name in top_items:
        new_features_dict[f'{col}_{item_name}'] = df_train_local[col].str.contains(item_name, regex=False, na=False).astype('int8')
    
    cols_to_drop.append(col)

df_new_features = pd.DataFrame(new_features_dict, index=df_train_local.index)
df_train_local = pd.concat([df_train_local, df_new_features], axis=1)
df_train_local.drop(columns=cols_to_drop, inplace=True, errors='ignore')

print(f"Filmes finais para treinamento: {len(df_train_local)}")
print(f"Número total de features estruturadas: {len(df_train_local.columns) - 1}")

X = df_train_local.drop('vote_average', axis=1)
y = df_train_local['vote_average']

features_for_prediction_final = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

del df_clean, df_train_local, X, y, df_new_features, new_features_dict
gc.collect()

model = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=5,
    max_features=0.4,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)

print(f"\n--- Avaliação do Modelo Random Forest para Previsão de Notas (Vote Average) ---")
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:,.2f} pontos na nota")
print(f"MAE: {mae:,.2f} pontos na nota")
print(f"MAPE: {mape * 100:.2f}%")

print("\n--- Top 20 Importância das Features ---")
feature_importances = sorted(zip(features_for_prediction_final, model.feature_importances_), key=lambda x: x[1], reverse=True)
for feat, imp in feature_importances[:20]:
    print(f"{feat}: {imp:.4f}")

with gzip.open('models/random_forest_vote_average_model.pkl.gz', 'wb') as f:
    pickle.dump(model, f)
    
with open('models/features_rf_vote_average_model.pkl', 'wb') as f:
    pickle.dump(features_for_prediction_final, f)

print("\nModelo e lista de features salvos com sucesso!")

Filmes com relevância estatística para treinamento (>= 5 votos): 125089
Filmes finais para treinamento: 125089
Número total de features estruturadas: 136

--- Avaliação do Modelo Random Forest para Previsão de Notas (Vote Average) ---
R² Score: 0.3733
RMSE: 0.89 pontos na nota
MAE: 0.67 pontos na nota
MAPE: 12.54%

--- Top 20 Importância das Features ---
genre_decade_vote_mean: 0.1798
runtime: 0.1309
vote_count: 0.0857
log_vote_count: 0.0790
votes_per_year: 0.0638
genre_vote_mean: 0.0555
genres_Horror: 0.0511
overview_len: 0.0507
movie_age: 0.0395
release_year: 0.0390
is_en: 0.0268
tagline_len: 0.0250
release_5_years: 0.0147
genres_Drama: 0.0130
production_countries_United States of America: 0.0118
budget: 0.0106
genres_Documentary: 0.0087
genres_Thriller: 0.0068
genres_Action: 0.0061
genres_Science Fiction: 0.0056

Modelo e lista de features salvos com sucesso!


In [3]:
import pickle
import shap
import numpy as np
import pandas as pd
from pathlib import Path

explainer_rf_vote = shap.TreeExplainer(model)
X_test_sampled = X_test.sample(n=500, random_state=42)
shap_values_rf_vote = explainer_rf_vote(X_test_sampled)

mean_abs_shap_rf_vote = np.abs(shap_values_rf_vote.values).mean(axis=0)

df_shap_imp_rf_vote = pd.DataFrame({
    'Feature': features_for_prediction_final,
    'Importance': mean_abs_shap_rf_vote
}).sort_values(by='Importance', ascending=False)

df_shap_imp_rf_vote.to_csv('models/shap/rf_vote_importance.csv', index=False)

shap_data_rf_vote = {
    'features': features_for_prediction_final,
    'shap_values': shap_values_rf_vote.values,
    'feature_values': X_test_sampled.values
}
with open('models/shap/rf_vote_shap_data.pkl', 'wb') as f:
    pickle.dump(shap_data_rf_vote, f)

c:\Users\dhavi\OneDrive\Área de Trabalho\Dhavi\UFRPE\3° Período\Projeto Interdisciplinar para Sistemas de Informação III\Projeto_PISI3_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
